In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 18
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 18
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [1]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 90 \
    --horizon 15 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99


🧹 Запуск модуля очистки данных (Сплиты, Иглы, Выбросы)...
Корректировка сплитов:  13%|██▊                  | 9/68 [00:00<00:00, 87.81it/s]  🕵️‍♂️ [HEURISTIC SPLIT] LSNGP@MISX на 2005-08-03: Коэфф 5.0
  📌 [KNOWN SPLIT] TRNFP@MISX на 2024-02-21: Коэфф 0.01
Корректировка сплитов:  43%|████████▌           | 29/68 [00:00<00:00, 94.53it/s]  🕵️‍♂️ [HEURISTIC SPLIT] AFKS@MISX на 2014-12-18: Коэфф 2.0
  🕵️‍♂️ [HEURISTIC SPLIT] NMTP@MISX на 2009-01-11: Коэфф 0.5
Корректировка сплитов: 100%|███████████████████| 68/68 [00:00<00:00, 107.55it/s]
✅ Очистка завершена!

🔄 [fold_2010/train - Build] Запуск...
✅ Готово: сохранено в /home/restorator/trader_test/data/processed/2000_2026_1d_90_15/fold_2010/data/train/dataset.csv

🔄 [fold_2010/train - Labels] Запуск...
✅ Авто-уровни: TP=14.80%, SL=12.21%
Разметка: 100%|████████████████████████████████| 39/39 [00:00<00:00, 187.82it/s]
🎉 Размеченный датасет сохранен: labels.csv

🔄 [fold_2010/train - Features] Запуск...

⚙️ [TRAIN] Инициализация расчета...
Этап

In [12]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

I0000 00:00:1777105622.449509   55864 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777105623.406142   55864 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
E0000 00:00:1777105624.395800   55864 cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (100)
✅ Расширенный аудит завершен: /home/restorator/trader_test/data_audit_report.txt


In [2]:
!python run_walkforward.py \
    --dataset_dir "data/processed/2000_2026_1d_90_15" \
    --runs 100 \
    --batch_size 4096 \
    --epochs 100 \
    --l2_reg "1e-3" \
    --lr "5e-2" \
    --start_fold "fold_2010" \
    --append

🚀 Запуск массового обучения моделей (Walk-Forward)...
📁 Датасет: data/processed/2000_2026_1d_90_15
⚙️  Настройки: 100 runs, 100 epochs, batch 4096
⏭️ Пропускаем завершенные фолды. Начинаем строго с: fold_2010

🔥 Обучение нейросети для: fold_2010
✅ Mixed precision включена!
✅ Динамическое выделение видеопамяти включено!
🚀 Старт обучения. Фолд: [fold_2010]
📊 Форма данных: [Lookback: 90, Features: 67]
⚙️ Расчет идеальных весов классов...
   Баланс: SL(0)=8622, Hold(1)=20601, TP(2)=8546
   Веса:   SL(0)=1.46, Hold(1)=0.61, TP(2)=1.47
⏳ Подготовка конвейера данных...

--------------------------------------------------
🔄 ИТЕРАЦИЯ 1/100 (Лучшая точность сессии: 0.00%)
--------------------------------------------------
Epoch 1/100
10/10 - 22s - 2s/step - accuracy: 0.3773 - loss: 6.3900 - val_accuracy: 0.1417 - val_loss: 9.8848
Epoch 2/100
10/10 - 2s - 169ms/step - accuracy: 0.4246 - loss: 3.5872 - val_accuracy: 0.1513 - val_loss: 8.7601
Epoch 3/100
10/10 - 2s - 162ms/step - accuracy: 0.4412 - 

In [ ]:
#очистка наименне успешных ltsm моделей (остается топ 3)
!python -m _tools.clean_lstm_models

In [ ]:
!python -m _tools.generate_model_specs

In [ ]:
!python -m _tools.prepare_rl_env

In [22]:
import pandas as pd
import numpy as np

# Путь к нашему финальному ансамблю
file_path = "data/processed/2000_2026_1d/rl_env/environment_data.parquet"

print("⏳ Загрузка датасета...")
df = pd.read_parquet(file_path)

print(f"✅ Датасет загружен! Размер: {df.shape[0]} строк, {df.shape[1]} колонок\n")

# 1. Проверка на пропуски (ОЧЕНЬ ВАЖНО из-за пропущенных фолдов)
missing_data = df.isnull().sum()
cols_with_nans = missing_data[missing_data > 0]

if not cols_with_nans.empty:
    print("⚠️ ВНИМАНИЕ! Найдены пропуски (NaN) в колонках:")
    # Выводим топ-10 колонок с наибольшим числом пропусков
    display(cols_with_nans.sort_values(ascending=False).head(10))
    print("\nВозможно, потребуется заполнить их нулями: df.fillna(0, inplace=True)")
else:
    print("✅ Пропусков (NaN) нет! Скрипт ансамбля всё корректно обработал (вероятно, заполнил нулями).")

# 2. Выделяем колонки наших экспертов
expert_cols = [col for col in df.columns if any(prefix in col for prefix in ['c18_', 'c30_', 'c60_', 'rank'])]

print(f"\n🧠 Найдено {len(expert_cols)} колонок от нейросетей-экспертов.")

# 3. Смотрим статистику предсказаний экспертов (чтобы убедиться, что там вероятности от 0 до 1)
if expert_cols:
    print("\n📊 Статистика мнений консилиума (mean, min, max):")
    display(df[expert_cols].describe().T[['mean', 'min', 'max']].head(15))

# 4. Визуальный срез (последние 5 дней)
print("\n👀 Хвост датасета (как это увидит RL-агент на последних шагах):")
# Покажем базовые данные (дата, тикер, цена) и пару колонок экспертов
base_cols = [c for c in ['datetime', 'ticker', 'close', 'label'] if c in df.columns]
cols_to_show = base_cols + expert_cols[:5] 
display(df[cols_to_show].tail())

⏳ Загрузка датасета...
✅ Датасет загружен! Размер: 164705 строк, 192 колонок

✅ Пропусков (NaN) нет! Скрипт ансамбля всё корректно обработал (вероятно, заполнил нулями).

🧠 Найдено 30 колонок от нейросетей-экспертов.

📊 Статистика мнений консилиума (mean, min, max):


,mean,min,max
cs_rank_mom_60,-0.033705,-1.735316,1.669760
cs_rank_vol_20,-0.035151,-1.742265,1.673408
cs_rank_liquidity_20,-0.032950,-1.740352,1.673610
c60_m1_p0,0.268250,0.006189,0.894771
c60_m1_p1,0.406341,0.011529,0.966990
c60_m1_p2,0.325409,0.010595,0.944944
c60_m2_p0,0.301382,0.014363,0.921730
c60_m2_p1,0.364843,0.015614,0.942843
c60_m2_p2,0.333775,0.009831,0.946455
c60_m3_p0,0.310776,0.007518,0.894099



👀 Хвост датасета (как это увидит RL-агент на последних шагах):


,datetime,ticker,label,cs_rank_mom_60,cs_rank_vol_20,cs_rank_liquidity_20,c60_m1_p0,c60_m1_p1
164700,2025-12-15,YDEX@MISX,0.0,1.330646,-0.934173,1.616924,0.216076,0.502307
164701,2025-12-16,YDEX@MISX,0.0,0.821975,0.142871,1.560239,0.215146,0.504268
164702,2025-12-17,YDEX@MISX,0.0,0.935013,-1.444352,1.616924,0.207647,0.523614
164703,2025-12-18,YDEX@MISX,0.0,1.274127,0.539677,1.673610,0.223960,0.473494
164704,2025-12-19,YDEX@MISX,0.0,1.319151,0.208205,1.673610,0.216491,0.493390


In [ ]:
!python -m _tools.train_rllib_pbt --population 6 --iterations 3000 --force

In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ray.rllib.algorithms.algorithm import Algorithm
from ray.tune.registry import register_env

# 1. Указываем Python, где искать класс твоей среды
sys.path.append(os.path.abspath('_tools'))

# 2. ИМПОРТ ТВОЕЙ СРЕДЫ
# Убедись, что файл называется train_rllib_pbt.py и класс называется TradingEnv
from train_rllib_pbt import TradingEnv 


def env_creator(env_config):
    # ИСПРАВЛЕНО: Распаковываем словарь, как этого ожидает твой класс
    return TradingEnv(**env_config)

def main():
    # === РЕГИСТРАЦИЯ СРЕДЫ ===
    # Обязательно делаем это ДО загрузки чекпоинта!
    register_env("TradingEnv-v0", env_creator)
    
    # 3. Автоматический поиск свежего чекпоинта
    base_dir = "data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot"
    
    # Ищем Trial 00004 (он показал +8% на тесте)
    trial_id = "00004" 
    trial_folders = glob.glob(f"{base_dir}/*_{trial_id}_*")

    if not trial_folders:
        raise FileNotFoundError(f"❌ Папка для Trial {trial_id} не найдена в {base_dir}!")

    trial_dir = trial_folders[0]

    # Ищем все чекпоинты внутри этой папки
    checkpoints = sorted(glob.glob(f"{trial_dir}/checkpoint_*"))

    if not checkpoints:
        raise FileNotFoundError(f"❌ В папке {trial_dir} нет сохраненных чекпоинтов!")

    # Берем самый последний (с максимальным номером) и делаем путь АБСОЛЮТНЫМ!
    CHECKPOINT_PATH = os.path.abspath(checkpoints[-1])
    print(f"✅ Найден чекпоинт:\n{CHECKPOINT_PATH}\n")

    # 4. Загрузка алгоритма
    print("⏳ Загрузка агента (это может занять около минуты)...")
    algo = Algorithm.from_checkpoint(CHECKPOINT_PATH)
    print("✅ Агент успешно загружен!")

    # 5. Инициализация среды строго в режиме TEST (экзамен 2022-2024)
    env_config = {
        "data_path": "data/processed/2000_2026_1d/rl_env/environment_data.parquet",
        "split_mode": "test", 
        "initial_balance": 100000.0,
        "commission": 0.0003,
        "max_episode_steps": 252 
    }
    
    # Создаем локальный экземпляр среды
    # ИСПРАВЛЕНО: Распаковываем словарь
    env = TradingEnv(**env_config)
    
    # 6. Прогон агента (Rollout)
    obs, info = env.reset()
    terminated = truncated = False
    history = []

    print("🤖 Агент начинает торговлю на тестовых данных...")
    
    while not (terminated or truncated):
        # Агент принимает детерминированное решение (explore=False) - строго то, чему научился
        action = algo.compute_single_action(obs, explore=False)
        
        # Делаем шаг в среде
        next_obs, reward, terminated, truncated, info = env.step(action)
        
        # Сохраняем историю для графиков
        history.append({
            'step': getattr(env, 'current_step', len(history)),
            'price': info.get('current_price', 0),
            'action': action,
            'balance': info.get('portfolio_value', 0),
            'reward': reward
        })
        
        obs = next_obs

    df_history = pd.DataFrame(history)
    print(f"\n🎉 Бэктест завершен! Итоговый баланс: ${df_history['balance'].iloc[-1]:.2f}")

    # ==========================================
    # 7. ВИЗУАЛИЗАЦИЯ: Графики цены и эквити
    # ==========================================
    print("📈 Построение графика...")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), gridspec_kw={'height_ratios': [3, 1]})

    # График цены
    ax1.plot(df_history['step'], df_history['price'], label='Цена актива', color='black', alpha=0.6)

    # Точки входа и выхода. 
    # ВНИМАНИЕ: Проверь логику своей среды! Здесь предполагается: 1 = Покупка, 2 = Продажа.
    buys = df_history[df_history['action'] == 1]
    sells = df_history[df_history['action'] == 2]

    ax1.scatter(buys['step'], buys['price'], marker='^', color='green', s=100, label='Покупка (Buy)', zorder=5)
    ax1.scatter(sells['step'], sells['price'], marker='v', color='red', s=100, label='Продажа (Sell)', zorder=5)

    ax1.set_title(f"Сделки RL-агента (Trial {trial_id}) на тестовых данных (2022-2024)")
    ax1.set_ylabel("Цена актива")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # График кривой доходности (Эквити)
    ax2.plot(df_history['step'], df_history['balance'], label='Кривая баланса (Equity)', color='blue', linewidth=2)
    ax2.axhline(y=100000.0, color='red', linestyle='--', alpha=0.5, label='Стартовый баланс') 
    
    ax2.set_ylabel("Баланс ($)")
    ax2.set_xlabel("Шаг среды")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()